## Setup -- select an AI provider and test the connection

This notebook can talk to a model through three different backends, all configured from a single shared module: `code/_shared/ai_config.py`. Credentials come from a `.env` file at the repo root (copy `.env.example` to `.env` and fill in whichever provider you actually have).

Pick one of `'gemini'`, `'bedrock'`, or `'anthropic'` below, then run the cell -- it sends one tiny real request and prints PASS, or a specific, actionable error if something is misconfigured. This step is optional: the rest of the notebook runs entirely in simulation mode either way, so a failed or skipped connection test does not block anything below.

In [ ]:
import sys
from pathlib import Path


def _add_shared_to_path() -> None:
    '''
    Locate code/_shared/ai_config.py by walking up from the current
    working directory, so this works whether the notebook is opened from
    template/, solutions/, or solved/.
    '''
    here = Path.cwd()
    for parent in [here, *here.parents]:
        candidate = parent / 'code' / '_shared'
        if (candidate / 'ai_config.py').is_file():
            if str(candidate) not in sys.path:
                sys.path.insert(0, str(candidate))
            return
    raise RuntimeError(
        'Could not find code/_shared/ai_config.py by walking up from the '
        'current working directory -- make sure this notebook is being run '
        'from inside the B04-AI_Agents folder tree.'
    )


_add_shared_to_path()
import ai_config

SELECTED_PROVIDER = 'gemini'  # change to 'bedrock' or 'anthropic' to try a different backend

try:
    provider = ai_config.get_provider(SELECTED_PROVIDER)
    ai_config.test_connection(provider)
except ai_config.AIConnectionError as exc:
    print(f'Connection check FAILED for provider {SELECTED_PROVIDER!r}:')
    print(f'  {exc}')
    print()
    print('The rest of this notebook still works fully in simulation mode --')
    print('this cell only matters for the optional real-API sections later on.')


# Chapter 1 -- The 2026 Agent Landscape (Practice)

Work through this notebook **after reading** `notes/ch01-agent-landscape.md`, especially Section 11's dry-run. Two exercises below have a stub for you to fill in, each followed by a **verification cell** you run to grade yourself. Everything else (config, plotting, the optional real-API comparison) is given and fully working.

This notebook reproduces Section 11's hand computation in code: instrumenting one 15-step agent-loop call, tracking how the fixed prefix (system prompt + tool schemas) and the growing conversation tail separately drive cost, with and without prompt caching.

In [ ]:
# ============================================================
# TOPIC: Instrumenting a single agent-loop call -- token growth
#        and prompt-caching economics across a 15-step loop
# MATH:  prefill_k = fixed_prefix + (k - 1) * (completion + observation)
# REF:   B04-AI_Agents notes/ch01-agent-landscape.md, Section 11
# ============================================================

# --- Imports (stdlib -> third-party -> local) ---
import os
from dataclasses import dataclass
from typing import List, Dict

import matplotlib.pyplot as plt
from dotenv import load_dotenv

# Load secrets from a local .env file (never committed -- see .env.example
# at the repo root). Safe to call even if .env does not exist yet.
load_dotenv()

# --- Constants / Config ---
# The exact scenario walked through by hand in notes Section 11.
# Nothing here is random -- the goal is to reproduce that hand
# arithmetic in code, then push past it with a real plot.

SYSTEM_PROMPT_TOKENS = 3_000     # durable instructions, sent every call
NUM_TOOLS = 6                    # tool schemas exposed to the model
TOOL_SCHEMA_TOKENS_EACH = 200    # ~200 tokens per tool schema (Ch 3 territory)
OBSERVATION_TOKENS_AVG = 800     # average tool-result size fed back in
COMPLETION_TOKENS_AVG = 150      # average thought-plus-tool-call the model emits
N_STEPS = 15                     # length of the agent loop (Ch 2's mini-agent)

FIXED_PREFIX_TOKENS = SYSTEM_PROMPT_TOKENS + NUM_TOOLS * TOOL_SCHEMA_TOKENS_EACH
TURN_GROWTH_TOKENS = COMPLETION_TOKENS_AVG + OBSERVATION_TOKENS_AVG

print('=' * 60)
print('CONFIG: one agent-loop run')
print('=' * 60)
print(f'  Fixed prefix (system + {NUM_TOOLS} tool schemas): {FIXED_PREFIX_TOKENS:,} tokens')
print(f'  Growth per step (completion + observation):        {TURN_GROWTH_TOKENS:,} tokens')
print(f'  Loop length:                                       {N_STEPS} steps')


In [ ]:
# --- Pricing (illustrative Sonnet-class rates, matching the notes) ---
# ALWAYS check current published rates before using these for a real
# budget -- prices change. These are the exact figures notes Section 11 uses.

INPUT_PRICE_PER_M_TOKENS = 3.00    # dollars per 1,000,000 input tokens (no cache)
OUTPUT_PRICE_PER_M_TOKENS = 15.00  # dollars per 1,000,000 output tokens
CACHE_WRITE_MULTIPLIER = 1.25      # 5-minute cache write
CACHE_READ_MULTIPLIER = 0.10       # cache read = 90 percent discount

CACHE_WRITE_PRICE_PER_M_TOKENS = INPUT_PRICE_PER_M_TOKENS * CACHE_WRITE_MULTIPLIER
CACHE_READ_PRICE_PER_M_TOKENS = INPUT_PRICE_PER_M_TOKENS * CACHE_READ_MULTIPLIER


def usd(tokens: float, price_per_m: float) -> float:
    '''
    Convert a token count into a dollar cost at a given per-million-token rate.
    Args: tokens -- number of tokens billed. price_per_m -- USD per 1,000,000 tokens.
    Returns: cost in USD.
    Math: cost = tokens * (price_per_m / 1_000_000)
    '''
    return tokens * (price_per_m / 1_000_000)


print(f'  Standard input:  ${INPUT_PRICE_PER_M_TOKENS:.2f} / 1M tokens')
print(f'  Cache write:     ${CACHE_WRITE_PRICE_PER_M_TOKENS:.2f} / 1M tokens (1.25x)')
print(f'  Cache read:      ${CACHE_READ_PRICE_PER_M_TOKENS:.2f} / 1M tokens (0.10x)')
print(f'  Output:          ${OUTPUT_PRICE_PER_M_TOKENS:.2f} / 1M tokens')


## Exercise 1 -- Implement the cached-cost branch of `simulate_step`

The stub below already computes, for step `k`: the fixed-prefix tokens, the growing-tail tokens, the total prefill, and `cost_no_cache` (everything priced at the standard input rate). Your job is to fill in `cost_with_cache` following notes Section 11's logic exactly:

- On **step 1**, the fixed prefix has never been seen before, so it is a **cache WRITE** (priced at `CACHE_WRITE_PRICE_PER_M_TOKENS`).
- On **every step after that**, the fixed prefix is a **cache READ** (priced at `CACHE_READ_PRICE_PER_M_TOKENS`).
- The growing tail is **never cacheable** -- it changes every step -- so it always costs the standard input rate.
- Do not forget the completion tokens, priced at the output rate, in both branches.

Replace the `# TODO` block inside `simulate_step` below.

In [ ]:
@dataclass
class StepRecord:
    '''One agent-loop step token and cost breakdown.'''
    step: int
    fixed_prefix_tokens: int
    tail_tokens: int
    prefill_tokens: int
    completion_tokens: int
    cost_no_cache: float
    cost_with_cache: float
    cache_event: str  # 'WRITE' or 'READ'


def simulate_step(step: int) -> StepRecord:
    '''
    Compute the token and cost breakdown for step `step` (1-indexed) of
    the N_STEPS-long agent loop, matching notes Section 11's derivation:
    prefill_k = FIXED_PREFIX_TOKENS + (k - 1) * TURN_GROWTH_TOKENS.
    '''
    prior_turns = step - 1
    tail_tokens = prior_turns * TURN_GROWTH_TOKENS
    prefill_tokens = FIXED_PREFIX_TOKENS + tail_tokens

    # --- cost WITHOUT caching: everything at the standard input rate (given) ---
    cost_no_cache = (
        usd(prefill_tokens, INPUT_PRICE_PER_M_TOKENS)
        + usd(COMPLETION_TOKENS_AVG, OUTPUT_PRICE_PER_M_TOKENS)
    )

    # --- cost WITH caching: TODO -- fill this in ---
    # cache_event = 'WRITE' if step == 1 else 'READ'
    # prefix_cost = ... (use CACHE_WRITE_PRICE_PER_M_TOKENS on step 1,
    #                     CACHE_READ_PRICE_PER_M_TOKENS otherwise)
    # tail_cost = ... (tail_tokens are NEVER cacheable -- standard input rate)
    # cost_with_cache = prefix_cost + tail_cost + (completion at output rate)
    cache_event = None       # TODO
    cost_with_cache = None   # TODO

    return StepRecord(
        step=step,
        fixed_prefix_tokens=FIXED_PREFIX_TOKENS,
        tail_tokens=tail_tokens,
        prefill_tokens=prefill_tokens,
        completion_tokens=COMPLETION_TOKENS_AVG,
        cost_no_cache=cost_no_cache,
        cost_with_cache=cost_with_cache,
        cache_event=cache_event,
    )


**Verification -- Exercise 1**

Run the full 15-step loop through your `simulate_step` and check the totals against notes Section 11's hand-derived numbers.

In [ ]:
print('=' * 60)
print('RUNNING THE 15-STEP AGENT LOOP (simulated)')
print('=' * 60)

records: List[StepRecord] = []
cumulative_no_cache = 0.0
cumulative_with_cache = 0.0

for k in range(1, N_STEPS + 1):
    rec = simulate_step(k)
    records.append(rec)
    cumulative_no_cache += rec.cost_no_cache
    cumulative_with_cache += rec.cost_with_cache

    print(f'\nSTEP {rec.step:2d} | cache {str(rec.cache_event):5s} | '
          f'prefill={rec.prefill_tokens:6,} tok '
          f'(prefix={rec.fixed_prefix_tokens:,} + tail={rec.tail_tokens:,}) | '
          f'completion={rec.completion_tokens} tok')
    print(f'         no-cache step cost=${rec.cost_no_cache:.6f} | '
          f'with-cache step cost=${rec.cost_with_cache:.6f}')
    print(f'         cumulative: no-cache=${cumulative_no_cache:.4f} | '
          f'with-cache=${cumulative_with_cache:.4f}')

total_prefill_tokens = sum(r.prefill_tokens for r in records)
total_completion_tokens = sum(r.completion_tokens for r in records)

print('\n' + '=' * 60)
print('SUMMARY -- 15-step run')
print('=' * 60)
print(f'  Total prefill tokens processed:  {total_prefill_tokens:,}')
print(f'  Total completion tokens:         {total_completion_tokens:,}')
print(f'  Total cost WITHOUT caching:      ${cumulative_no_cache:.4f}')
print(f'  Total cost WITH caching:         ${cumulative_with_cache:.4f}')

# --- Cross-check against the hand computation in notes Section 11 ---
EXPECTED_TOTAL_PREFILL = 162_750
EXPECTED_NO_CACHE_COST = 0.522
EXPECTED_WITH_CACHE_COST = 0.36639

assert total_prefill_tokens == EXPECTED_TOTAL_PREFILL, 'Prefill token total drifted from the notes derivation.'
assert abs(cumulative_no_cache - EXPECTED_NO_CACHE_COST) < 0.001, 'No-cache cost drifted from the notes.'
assert abs(cumulative_with_cache - EXPECTED_WITH_CACHE_COST) < 0.001, 'Cached cost drifted from the notes -- check your Exercise 1 branch.'
print('\nPASS -- matches the hand-derived arithmetic in notes/ch01-agent-landscape.md Section 11 exactly.')


## Exercise 2 -- Why does not caching save close to 90 percent?

Cache reads are a 90 percent discount. Yet the full-run savings you just measured are nowhere near 90 percent. Compute `savings_pct` below from the two totals you already have, then write one sentence in `my_explanation` naming which part of the context the 90 percent discount can never touch.

In [ ]:
# TODO: compute the percentage saved by caching, using the two totals above
savings_pct = None  # TODO: (cumulative_no_cache - cumulative_with_cache) / cumulative_no_cache * 100

# TODO: one sentence -- which part of the context can the cache discount never reach?
my_explanation = ''  # TODO


**Verification -- Exercise 2**

In [ ]:
assert savings_pct is not None, 'Fill in savings_pct first.'
assert 20 < savings_pct < 40, f'Expected roughly 25 to 35 percent savings, got {savings_pct:.1f} percent -- recheck the formula.'
assert len(my_explanation.strip()) > 0, 'Write your one-sentence explanation in my_explanation.'
print(f'PASS -- caching saved {savings_pct:.1f} percent on this run.')
print(f'Your explanation: {my_explanation}')
print('\n(Compare against notes Section 11: the growing, uncacheable tail')
print('dominates the bill, so an 84 percent discount on the fixed prefix alone')
print('only becomes a roughly 30 percent discount on the whole run.)')


### Everything below this line is given -- no exercises

A plot of cumulative cost, with and without caching, across all 15 steps.

In [ ]:
steps = [r.step for r in records]
cum_no_cache, cum_cache = [], []
running_no, running_yes = 0.0, 0.0
for r in records:
    running_no += r.cost_no_cache
    running_yes += r.cost_with_cache
    cum_no_cache.append(running_no)
    cum_cache.append(running_yes)

plt.figure(figsize=(8, 5))
plt.plot(steps, cum_no_cache, marker='o', label='Cumulative cost -- no caching')
plt.plot(steps, cum_cache, marker='o', label='Cumulative cost -- with prompt caching')
plt.xlabel('Agent loop step')
plt.ylabel('Cumulative cost (USD)')
plt.title('Why caching helps less than expected: the growing tail dominates')
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.savefig('ch01_cost_growth.png', dpi=120)
plt.show()

print('\nNotice the two lines are roughly PARALLEL after the first couple of')
print('steps, not converging -- caching gives a one-time discount on the')
print('fixed prefix, but both lines keep climbing at almost the same rate,')
print('driven by the uncacheable growing tail. This is the arithmetic')
print('argument for context management, covered in Chapter 4, not just a caching tip.')


### Optional -- compare against REAL numbers from whichever provider you selected

This section reuses the exact same `ai_config` import and `SELECTED_PROVIDER` choice from the setup cell at the top of this notebook -- it is not hardcoded to any one company. Flip `RUN_REAL_API_DEMO` to `True` to run a few real steps and see measured token/cache numbers instead of the simulated numbers above.

Real prompt-caching numbers (`cache_read_tokens` / `cache_write_tokens`) only show up for `'anthropic'` and `'bedrock'`, since both support a real cache checkpoint on the system block. Gemini accepts a system instruction here too, but does not cache it this way -- real Gemini caching needs a separate `CachedContent` resource that this shared client does not create, so its cache fields always read `None`.

In [ ]:
# --- Optional: run a few REAL steps against whichever provider SELECTED_PROVIDER
# points to (set in the setup cell at the top) -- via the shared ai_config
# module, not a hardcoded per-company SDK call.

RUN_REAL_API_DEMO = False
REAL_DEMO_STEPS = 4  # keep small -- this costs real money/quota


def run_real_api_demo(n_steps: int = REAL_DEMO_STEPS) -> None:
    """
    Run a few steps of the same growing-conversation scenario against
    SELECTED_PROVIDER, printing ACTUAL usage numbers from ai_config's
    normalized usage dict -- so you can compare real measured
    tokens/cache activity to the simulated numbers earlier in this notebook.
    """
    try:
        provider = ai_config.get_provider(SELECTED_PROVIDER)
    except ai_config.AIConnectionError as exc:
        print(f"Skipping real API demo: {exc}")
        return

    # Comfortably over the largest known per-model caching minimum (4,096 tokens).
    system_text = "You are a terse test agent. " * 600

    print("-" * 60)
    print(f"RUNNING {n_steps} REAL STEPS AGAINST {SELECTED_PROVIDER!r} (model={provider.model})")
    print("-" * 60)

    for k in range(1, n_steps + 1):
        try:
            result = provider.generate(
                f"Step {k}: continue the task in one short sentence.",
                system=system_text,
            )
        except ai_config.AIConnectionError as exc:
            print(f"STEP {k} FAILED: {exc}")
            return

        u = result.usage
        print(f"\nSTEP {k} (REAL, {SELECTED_PROVIDER}) | input_tokens={u['input_tokens']} | "
              f"output_tokens={u['output_tokens']} | "
              f"cache_read_tokens={u['cache_read_tokens']} | "
              f"cache_write_tokens={u['cache_write_tokens']}")

        if SELECTED_PROVIDER == "gemini":
            if k == 1:
                print("         (Gemini cache fields are always None here -- this is expected, not a problem;")
                print("          see this function's docstring -- Gemini needs a separate CachedContent setup)")
        elif k > 1 and not u["cache_read_tokens"] and not u["cache_write_tokens"]:
            print("         (both cache fields are 0 -- system_text may be below this model's minimum cacheable length)")


if RUN_REAL_API_DEMO:
    run_real_api_demo()
else:
    print("RUN_REAL_API_DEMO is False -- running in simulation-only mode.")
    print(f"Flip it to True to run {REAL_DEMO_STEPS} real steps against ")
    print(f"SELECTED_PROVIDER ({SELECTED_PROVIDER!r}, set in the setup cell above)")
    print("and compare real measured cache/token numbers to the simulated ones.")


## Key Takeaways

A 15-step agent loop processes over 160,000 prefill tokens because the whole transcript resends every step. Prompt caching gives a real discount, but only on the stable fixed prefix -- the growing conversation tail can never be cached, and it is what actually dominates the bill on a real run. That is the arithmetic case for Chapter 4's context management: caching is a discount on repetition, not a cure for growth.

**Connection forward:** `ch02` stops talking about the loop and writes one -- a real, working agent in plain Python, no framework.